<a href="https://colab.research.google.com/github/muhammeddanisht/langgraph-agents/blob/main/langgraph_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V6 — MCP Agent (Real Tools via Model Context Protocol)

**What this is:** A LangGraph agent connected to real external tools via MCP
(Model Context Protocol). Tools run on a local FastMCP server and are called
through the official langchain-mcp-adapters bridge.

**Architecture:** FastMCP server (streamable-http) → MultiServerMCPClient →
create_agent (ReAct loop) → Groq LLM.

**Stack:** FastMCP 3.4.2 · langchain-mcp-adapters · LangGraph · Groq (llama-3.3-70b-versatile) · Python

**Tools:** calculator (AST-based safe evaluator, no eval()) · get_weather (live wttr.in data)

**Builds on:** V5 memory architecture — V6 adds real MCP tool integration.

Install Packages

In [2]:
!pip install -q fastmcp langchain-mcp-adapters langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.2/221.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.9/272.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requir

 Imports

In [3]:
# MCP server builder
from fastmcp import FastMCP

# Background thread for Colab
import threading
import time

# Real weather data
import requests

# Safe calculator (replaces eval)
import ast
import operator

# Groq LLM
from langchain_groq import ChatGroq

# MCP client
from langchain_mcp_adapters.client import MultiServerMCPClient

# REMOVED: create_react_agent (deprecated, never used)
# create_agent imported where needed inside build_agent only

# Message format
from langchain_core.messages import HumanMessage

# asyncio fix for Colab
import asyncio
import nest_asyncio
nest_asyncio.apply()

import os
from google.colab import userdata

Get API Key from Colab Secrets

In [5]:
import os
from google.colab import userdata

# Read key from Colab secrets safe
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ API key loaded from Colab secrets!")

✅ API key loaded from Colab secrets!


 Build MCP Server with Real Tools

In [6]:
# Step 1 — Create the MCP server
# Give it a name — "my_tools"
mcp = FastMCP("my_tools")

Calculator Tool

In [7]:
SAFE_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.FloorDiv: operator.floordiv,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def safe_eval(expression: str):
    tree = ast.parse(expression, mode='eval')
    return _eval_node(tree.body)

def _eval_node(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numbers allowed")
    elif isinstance(node, ast.BinOp):
        op_type = type(node.op)
        if op_type not in SAFE_OPERATORS:
            raise ValueError(f"Operator not allowed: {op_type.__name__}")
        return SAFE_OPERATORS[op_type](_eval_node(node.left), _eval_node(node.right))
    elif isinstance(node, ast.UnaryOp):
        op_type = type(node.op)
        if op_type not in SAFE_OPERATORS:
            raise ValueError(f"Operator not allowed: {op_type.__name__}")
        return SAFE_OPERATORS[op_type](_eval_node(node.operand))
    else:
        raise ValueError(f"Not allowed: {type(node).__name__}")

@mcp.tool()
def calculator(expression: str) -> str:
    """Calculate a math expression. Example: '25 * 4' or '100 / 5 + 3'"""
    try:
        result = safe_eval(expression)   # FIXED: no eval()
        return f"The result of {expression} = {result}"
    except Exception as e:
        return f"Math error: {str(e)}"

 Weather Tool (real live data from wttr.in)

In [8]:
@mcp.tool()
def get_weather(city: str) -> str:
    """
    Get real current weather for any city in the world.
    Example: 'London' or 'Tokyo' or 'Kozhikode'
    """
    try:
        # wttr.in = free weather service, no API key needed
        # format=3 gives us: "City: emoji temperature"
        url = f"https://wttr.in/{city}?format=3"
        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            return f"Current weather → {response.text.strip()}"
        else:
            return f"Could not fetch weather for {city}"
    except Exception as e:
        return f"Weather error: {str(e)}"

 Start server in background thread (Colab fix)

In [9]:
# daemon=True means: when notebook stops, server stops too

def run_server():
    mcp.run(transport="streamable-http")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give server 2 seconds to fully start before we connect
time.sleep(2)

print("✅ MCP Server running in background!")
print("📡 Listening at: http://localhost:8000/mcp")

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.4.2                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      my_tools, 3.4.2                             │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[06/19/26 05:52:39] INFO     Starting MCP server 'my_tools' with transport 'streamable-http' on    ]8;id=213165;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=477888;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#304\304]8;;\
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [1423]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


✅ MCP Server running in background!
📡 Listening at: http://localhost:8000/mcp


Build the LLM (Groq free model)

In [10]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("✅ Groq LLM ready!")

✅ Groq LLM ready!


  asyncio for Colab

In [11]:

!pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()

print("✅ asyncio Colab fix applied!")

✅ asyncio Colab fix applied!


Connect Agent to MCP Server (Modern 2026 way)


In [12]:
from langchain.agents import create_agent

async def build_agent():
    client = MultiServerMCPClient(
        {
            "my_tools": {
                "transport": "http",
                "url": "http://localhost:8000/mcp"
            }
        }
    )

    tools = await client.get_tools()

    # Modern 2026 way — no deprecation warning
    agent = create_agent(
        model=llm,
        tools=tools
    )

    return agent, tools

agent, tools = asyncio.run(build_agent())

print(f"✅ Agent ready!")
print(f"🔧 Tools loaded: {[t.name for t in tools]}")

INFO:     127.0.0.1:58096 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58108 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58116 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58122 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58124 - "DELETE /mcp HTTP/1.1" 200 OK
✅ Agent ready!
🔧 Tools loaded: ['calculator', 'get_weather']


Test the Agent


In [13]:


import asyncio

async def ask_agent(question):
    print(f"\n❓ Question: {question}")
    print("-" * 50)

    response = await agent.ainvoke({
        "messages": [HumanMessage(content=question)]
    })

    # Get the last message — that's the final answer
    final_answer = response["messages"][-1].content
    print(f"🤖 Answer: {final_answer}")
    return final_answer

# Test 1 — Calculator (real math)
asyncio.run(ask_agent("What is 347 multiplied by 28?"))


❓ Question: What is 347 multiplied by 28?
--------------------------------------------------
INFO:     127.0.0.1:58138 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58140 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58156 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58172 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58182 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58186 - "DELETE /mcp HTTP/1.1" 200 OK
🤖 Answer: The result of the multiplication is 9716.


'The result of the multiplication is 9716.'

Test Weather Tool

In [14]:
asyncio.run(ask_agent("What is the current weather in Tokyo?"))


❓ Question: What is the current weather in Tokyo?
--------------------------------------------------
INFO:     127.0.0.1:58192 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58198 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58212 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58214 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58218 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58234 - "DELETE /mcp HTTP/1.1" 200 OK
🤖 Answer: The current weather in Tokyo is 79°F with a clear sky.


'The current weather in Tokyo is 79°F with a clear sky.'

In [15]:
# Boss Test (no hints given)

asyncio.run(ask_agent("I have 3 boxes. Each box has 144 items. How many items total?"))


❓ Question: I have 3 boxes. Each box has 144 items. How many items total?
--------------------------------------------------
INFO:     127.0.0.1:58238 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58250 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58252 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58268 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58276 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58282 - "DELETE /mcp HTTP/1.1" 200 OK
🤖 Answer: You have 432 items in total.


'You have 432 items in total.'

In [16]:
#— Another boss test

asyncio.run(ask_agent("What's the weather in Kozhikode right now?"))


❓ Question: What's the weather in Kozhikode right now?
--------------------------------------------------
INFO:     127.0.0.1:58294 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58310 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58314 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58330 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58342 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58356 - "DELETE /mcp HTTP/1.1" 200 OK
🤖 Answer: The current weather in Kozhikode is 84°F with a sunny sky.


'The current weather in Kozhikode is 84°F with a sunny sky.'